In [1]:
!python -m pip install -U setuptools pip
!pip install "cupy-cuda12x[ctk]"
!pip install cuda-cccl[cu13]

In [ ]:
!nvidia-smi

Sat Feb 28 00:51:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import numpy as np
import cupy as cp
import cuda.compute
import numpy as np
from numba import cuda, float32

# Sección nueva

In [3]:
import random
import time

def create_random_matrix(rows, cols):
    return [[random.randint(0, 10) for _ in range(cols)] for _ in range(rows)]

N = 1000
M = 5000

In [ ]:
%%timeit
def matmul(matrix_x, matrix_y):
    rows_a = len(matrix_x)
    cols_a = len(matrix_x[0])
    rows_b = len(matrix_y)
    cols_b = len(matrix_y[0])

    result = [[0 for _ in range(cols_b)] for _ in range(rows_a)]

    for i in range(rows_a):
        for j in range(cols_b):
            for k in range(cols_a):
                result[i][j] += matrix_x[i][k] * matrix_y[k][j]

    return result

matrix1 = create_random_matrix(N, M)

matrix2 = create_random_matrix(M, N)

result = matmul(matrix1, matrix2)

KeyboardInterrupt: 

In [ ]:
%%timeit
numpy_matrix1 = np.random.randint(0, 11, size=(N, M))
numpy_matrix2 = np.random.randint(0, 11, size=(M, N))

numpy_result = np.matmul(numpy_matrix1, numpy_matrix2)

KeyboardInterrupt: 

In [4]:
%%timeit
cupy_matrix1 = cp.random.randint(0, 11, size=(N, M))
cupy_matrix2 = cp.random.randint(0, 11, size=(M, N))

cupy_result = cp.matmul(cupy_matrix1, cupy_matrix2)

28.8 ms ± 3.78 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
%%timeit
@cuda.jit # Just-in-Time Compiler
def matmul_kernel(A, B, C):
    #row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    #col = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    row, col = cuda.grid(2)
    if row < C.shape[0] and col < C.shape[1]:
        tmp = 0.0
        for k in range(A.shape[1]):
            tmp += A[row, k] * B[k, col]
        C[row, col] = tmp

A = np.random.rand(N, M).astype(np.float32) # cpu
B = np.random.rand(M, N).astype(np.float32) # cpu
C = np.zeros((N, N), dtype=np.float32) # cpu

d_A = cuda.to_device(A) # GPU
d_B = cuda.to_device(B) # GPU
d_C = cuda.device_array((N, N), dtype=np.float32) # GPU

threadsperblock = (32, 32)
blockspergrid_x = int(np.ceil(C.shape[1] / threadsperblock[1]))
blockspergrid_y = int(np.ceil(C.shape[0] / threadsperblock[0]))
blockspergrid = (blockspergrid_x, blockspergrid_y)

matmul_kernel[blockspergrid, threadsperblock](d_A, d_B, d_C)

C = d_C.copy_to_host()

428 ms ± 59.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [7]:
@cuda.jit
def diff_kernel(pred: np.ndarray, target: np.ndarray, out_diff: np.ndarray) -> None:
    """Unfused step 1: out_diff = pred - target."""
    i = cuda.grid(1)
    if i < pred.size:
        out_diff[i] = pred[i] - target[i]


@cuda.jit
def square_kernel(inp: np.ndarray, out_sq: np.ndarray) -> None:
    """Unfused step 2: out_sq = inp * inp."""
    i = cuda.grid(1)
    if i < inp.size:
        out_sq[i] = inp[i] * inp[i]


@cuda.jit
def reduce_sum_kernel(inp: np.ndarray, out_sum: np.ndarray) -> None:
    """Unfused step 3: reduce by atomic adding a local running sum."""
    idx = cuda.grid(1)
    stride = cuda.gridsize(1)

    local_sum = np.float32(0.0)
    for i in range(idx, inp.size, stride):
        local_sum += inp[i]

    cuda.atomic.add(out_sum, 0, local_sum)

def mse_numba_unfused(pred: cp.ndarray, target: cp.ndarray, blocks: int, threads: int) -> float:
    """Numba unfused path: diff kernel -> square kernel -> reduce kernel."""
    pred_numba = cuda.as_cuda_array(pred)
    target_numba = cuda.as_cuda_array(target)

    out_diff = cuda.device_array(shape=pred.size, dtype=np.float32)
    out_sq = cuda.device_array(shape=pred.size, dtype=np.float32)
    out_sum = cuda.to_device(np.array([0.0], dtype=np.float32))

    diff_kernel[blocks, threads](pred_numba, target_numba, out_diff)
    square_kernel[blocks, threads](out_diff, out_sq)
    reduce_sum_kernel[blocks, threads](out_sq, out_sum)
    cuda.synchronize()

    mse_sum = float(out_sum.copy_to_host()[0])
    return mse_sum / pred.size

In [10]:
@cuda.jit
def fused_mse_sum_kernel(pred: np.ndarray, target: np.ndarray, out_sum: np.ndarray) -> None:
    """Compute sum((pred - target)^2) in one fused kernel.

    Each thread accumulates a local running sum over a strided range.
    Only one global write per thread is done (atomic add to out_sum[0]).
    """
    idx = cuda.grid(1)
    stride = cuda.gridsize(1)

    local_sum = np.float32(0.0)
    for i in range(idx, pred.size, stride):
        diff = pred[i] - target[i]
        local_sum += diff * diff

    cuda.atomic.add(out_sum, 0, local_sum)

def mse_numba_fused(pred: cp.ndarray, target: cp.ndarray, blocks: int, threads: int) -> float:
    pred_numba = cuda.as_cuda_array(pred)
    target_numba = cuda.as_cuda_array(target)
    out_sum = cuda.to_device(np.array([0.0], dtype=np.float32))

    fused_mse_sum_kernel[blocks, threads](pred_numba, target_numba, out_sum)
    cuda.synchronize()

    mse_sum = float(out_sum.copy_to_host()[0])
    return mse_sum / pred.size

In [8]:
cp.cuda.runtime.deviceSynchronize()
cuda.synchronize()
rng = cp.random.default_rng(42)
pred = rng.standard_normal(100000, dtype=np.float64)
target = rng.standard_normal(100000, dtype=np.float64)

In [9]:
%%timeit
mse_numba_unfused(pred, target, blocks=40*8, threads=256)

1.31 ms ± 545 µs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [11]:
%%timeit
mse_numba_fused(pred, target, blocks=40*8, threads=256)

839 µs ± 93.4 µs per loop (mean ± std. dev. of 7 runs, 1000 loops each)
